In [1]:
!pip install catboost lightgbm xgboost category_encoders pygeohash -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from category_encoders import TargetEncoder

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import pygeohash as pgh

import warnings
warnings.filterwarnings("ignore")

In [3]:
train = pd.read_csv("Dataset/train.csv")
test = pd.read_csv("Dataset/test.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [4]:
for col in ["RoadType","Weather"]:

    train[col] = train[col].fillna("Unknown")
    test[col] = test[col].fillna("Unknown")

median_temp = train["Temperature"].median()

train["Temperature"] = train["Temperature"].fillna(median_temp)
test["Temperature"] = test["Temperature"].fillna(median_temp)

In [5]:
def create_time_features(df):

    parts = df["timestamp"].str.split(
        ":",
        expand=True
    )

    df["hour"] = parts[0].astype(int)
    df["minute"] = parts[1].astype(int)

    df["hour_sin"] = np.sin(
        2*np.pi*df["hour"]/24
    )

    df["hour_cos"] = np.cos(
        2*np.pi*df["hour"]/24
    )

    df["minute_sin"] = np.sin(
        2*np.pi*df["minute"]/60
    )

    df["minute_cos"] = np.cos(
        2*np.pi*df["minute"]/60
    )

    df["is_peak"] = (
        ((df["hour"]>=7)&(df["hour"]<=10))
        |
        ((df["hour"]>=16)&(df["hour"]<=20))
    ).astype(int)

    return df

train = create_time_features(train)
test = create_time_features(test)

In [6]:
freq = train["geohash"].value_counts()

train["geo_freq"] = train["geohash"].map(freq)

test["geo_freq"] = (
    test["geohash"]
    .map(freq)
    .fillna(0)
)

In [7]:
geo_mean = train.groupby(
    "geohash"
)["demand"].mean()

train["geo_mean"] = (
    train["geohash"]
    .map(geo_mean)
)

test["geo_mean"] = (
    test["geohash"]
    .map(geo_mean)
)

In [8]:
def decode_geo(g):

    try:

        lat, lon = pgh.decode(g)

        return pd.Series(
            [lat, lon]
        )

    except:

        return pd.Series(
            [np.nan, np.nan]
        )

train[["lat","lon"]] = (
    train["geohash"]
    .apply(decode_geo)
)

test[["lat","lon"]] = (
    test["geohash"]
    .apply(decode_geo)
)

In [9]:
def decode_geo(g):

    try:

        lat, lon = pgh.decode(g)

        return pd.Series(
            [lat, lon]
        )

    except:

        return pd.Series(
            [np.nan, np.nan]
        )

train[["lat","lon"]] = (
    train["geohash"]
    .apply(decode_geo)
)

test[["lat","lon"]] = (
    test["geohash"]
    .apply(decode_geo)
)

In [10]:
train["geo_te"] = np.nan

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for tr_idx,val_idx in kf.split(train):

    te = TargetEncoder(
        cols=["geohash"]
    )

    te.fit(
        train.iloc[tr_idx][["geohash"]],
        train.iloc[tr_idx]["demand"]
    )

    train.loc[
        val_idx,
        "geo_te"
    ] = te.transform(
        train.iloc[val_idx][["geohash"]]
    ).values.ravel()

final_te = TargetEncoder(
    cols=["geohash"]
)

final_te.fit(
    train[["geohash"]],
    train["demand"]
)

test["geo_te"] = (
    final_te.transform(
        test[["geohash"]]
    ).values.ravel()
)

In [11]:
for df in [train,test]:

    df["temp_lane"] = (
        df["Temperature"]
        *
        df["NumberofLanes"]
    )

    df["temp_hour"] = (
        df["Temperature"]
        *
        df["hour"]
    )

    df["lane_hour"] = (
        df["NumberofLanes"]
        *
        df["hour"]
    )

    df["peak_temp"] = (
        df["is_peak"]
        *
        df["Temperature"]
    )

In [12]:
TARGET = "demand"

DROP_COLS = [
    "Index",
    "timestamp"
]

X = train.drop(
    columns=DROP_COLS+[TARGET]
)

y = train[TARGET]

X_test = test.drop(
    columns=DROP_COLS
)

In [13]:
cat_cols = [
    "geohash",
    "RoadType",
    "Weather",
    "LargeVehicles",
    "Landmarks"
]

for col in cat_cols:

    all_vals = pd.concat([
        X[col].astype(str),
        X_test[col].astype(str)
    ])

    mapping = {
        k:v
        for v,k in enumerate(
            all_vals.unique()
        )
    }

    X[col] = (
        X[col]
        .astype(str)
        .map(mapping)
    )

    X_test[col] = (
        X_test[col]
        .astype(str)
        .map(mapping)
    )

In [14]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pred_cat = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))

In [15]:
for tr_idx,val_idx in kf.split(X):

    model = CatBoostRegressor(
        iterations=7000,
        learning_rate=0.02,
        depth=12,
        loss_function="RMSE",
        verbose=0
    )

    model.fit(
        X.iloc[tr_idx],
        y.iloc[tr_idx]
    )

    pred_cat += (
        model.predict(X_test)
        / 5
    )

KeyboardInterrupt: 